In [16]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

In [17]:
tree_data = pd.read_csv(r'..\Data\new_york_tree_census_2015.csv')

In [18]:
tree_data_small = tree_data[['tree_dbh', 'curb_loc', 'health', 'spc_common', 'sidewalk', 'zipcode']].copy()
tree_data_small

,tree_dbh,curb_loc,health,spc_common,sidewalk,zipcode
0,10,OnCurb,Good,green ash,NoDamage,11366
1,9,OnCurb,Good,honeylocust,NoDamage,11370
2,7,OnCurb,Good,Callery pear,NoDamage,11434
3,10,OnCurb,Good,Callery pear,NoDamage,11209
4,4,OnCurb,Good,'Schubert' chokecherry,NoDamage,11692
...,...,...,...,...,...,...
683783,2,OnCurb,Poor,purple-leaf plum,NoDamage,11231
683784,2,OnCurb,NaN,NaN,NaN,11001
683785,2,OnCurb,NaN,NaN,NaN,11230
683786,18,OnCurb,Good,northern red oak,Damage,11420


In [19]:
# initial clean of the data - drop na
tree_data_small.dropna(inplace=True)

tree_data_small.info()

<class 'pandas.core.frame.DataFrame'>
Index: 652166 entries, 0 to 683787
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   tree_dbh    652166 non-null  int64 
 1   curb_loc    652166 non-null  object
 2   health      652166 non-null  object
 3   spc_common  652166 non-null  object
 4   sidewalk    652166 non-null  object
 5   zipcode     652166 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 34.8+ MB


In [20]:
def get_ratio(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.iloc[0]/ val_counts.iloc[1]
       # if there is only one value then the ratio is 1
       return 1
           
def get_first(series):
       val_counts = series.value_counts()
       return val_counts.index[0]

def get_second(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.index[1]
       else:
              return val_counts.index[-1]

def get_third(series):
       val_counts = series.value_counts()
       if len(val_counts) > 2:
              return val_counts.index[2]
       else:
              return val_counts.index[-1]
       
def count_unique_vals(series):
       return series.value_counts().to_dict()
       

In [21]:
# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
tree_data_agg = tree_data_small.groupby('zipcode').agg(
       dbh_mean=('tree_dbh', 'mean'),
       curb_ratio=('curb_loc', get_ratio),
       sidewalk_ratio=('sidewalk', get_ratio),
       health_counts=('health', count_unique_vals),
       species_1=('spc_common', get_first),
       species_2=('spc_common', get_second),
       species_3=('spc_common', get_third),
       tree_count=('tree_dbh', 'size')
    
).reset_index()

health_counts_df = pd.DataFrame(tree_data_agg['health_counts'].tolist()).fillna(0).astype(int)
tree_data_agg = pd.concat([tree_data_agg, health_counts_df], axis=1).drop(columns=['health_counts'])

tree_data_agg


,zipcode,dbh_mean,curb_ratio,sidewalk_ratio,species_1,species_2,species_3,tree_count,Good,Fair,Poor
0,83,16.505365,1.273171,3.217195,American elm,pin oak,ginkgo,932,885,39,8
1,10001,7.365882,46.222222,4.629139,honeylocust,Callery pear,Japanese zelkova,850,719,100,31
2,10002,8.459222,5.255072,4.678947,London planetree,honeylocust,ginkgo,2158,1656,387,115
3,10003,9.077200,91.523810,2.672968,honeylocust,Callery pear,ginkgo,1943,1480,361,102
4,10004,6.658120,3.178571,28.250000,honeylocust,ginkgo,Japanese zelkova,117,97,17,3
...,...,...,...,...,...,...,...,...,...,...,...
186,11691,8.813316,27.221053,4.432624,honeylocust,London planetree,cherry,5362,3761,1258,343
187,11692,4.962963,12.027972,5.900000,Callery pear,cherry,Sophora,1863,1122,523,218
188,11693,7.897335,6.914062,12.876712,London planetree,honeylocust,Callery pear,1013,518,288,207
189,11694,8.053841,17.358382,6.234624,honeylocust,Callery pear,cherry,3176,1958,833,385


In [22]:
# need to encode categorical data
encoder = OrdinalEncoder()

# encode the three species columns
tree_encoded_df = tree_data_agg
tree_encoded_df[['species_1', 'species_2', 'species_3']] = encoder.fit_transform(tree_data_agg[['species_1', 'species_2', 'species_3']])

tree_encoded_df

,zipcode,dbh_mean,curb_ratio,sidewalk_ratio,species_1,species_2,species_3,tree_count,Good,Fair,Poor
0,83,16.505365,1.273171,3.217195,0.0,13.0,10.0,932,885,39,8
1,10001,7.365882,46.222222,4.629139,7.0,2.0,3.0,850,719,100,31
2,10002,8.459222,5.255072,4.678947,3.0,10.0,10.0,2158,1656,387,115
3,10003,9.077200,91.523810,2.672968,7.0,2.0,10.0,1943,1480,361,102
4,10004,6.658120,3.178571,28.250000,7.0,8.0,3.0,117,97,17,3
...,...,...,...,...,...,...,...,...,...,...,...
186,11691,8.813316,27.221053,4.432624,7.0,4.0,9.0,5362,3761,1258,343
187,11692,4.962963,12.027972,5.900000,1.0,7.0,8.0,1863,1122,523,218
188,11693,7.897335,6.914062,12.876712,3.0,10.0,2.0,1013,518,288,207
189,11694,8.053841,17.358382,6.234624,7.0,2.0,9.0,3176,1958,833,385


In [23]:
crime_data = pd.read_csv(r'..\Data\2015_Crime.csv', low_memory=False)

In [24]:
crime_data.head()
# Keep only the specified columns
crime_data = crime_data[['Violation Date', 'Violation Time', 'Issuing Agency',
                         'Violation Location (Zip Code)', 
                         'Penalty Imposed', 'Charge #1: Code', 
                         'Charge #2: Code', 'Charge #3: Code', 'Charge #4: Code', 
                         'Charge #5: Code', 'Charge #6: Code', 'Charge #7: Code', 
                         'Charge #8: Code', 'Charge #9: Code', 'Charge #10: Code']]

# Remove rows where the 'Violation Location (Zip Code)' column is NaN
crime_data = crime_data.dropna(subset=['Violation Location (Zip Code)'])

crime_data = crime_data.dropna(subset=['Issuing Agency'])


In [25]:
# Extract columns containing charges
charge_columns = [col for col in crime_data.columns if "Charge" in col]

crime_data = crime_data.melt(
    id_vars=[col for col in crime_data.columns if col not in charge_columns],  # Keep these columns unchanged  # Columns to unpivot
    var_name="Original Charge Column",  # New column for original charge column names
    value_name="Charge: Code",  # New column for charge values
)

# Drop rows where Charge: Code is NaN
crime_data = crime_data.dropna(subset=["Charge: Code"]).drop(columns=["Original Charge Column"])

# Reset index for a clean DataFrame
crime_data = crime_data.reset_index(drop=True)

# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
crime_data_final = crime_data.groupby('Violation Location (Zip Code)').agg(
    issuing_agency_1=('Issuing Agency', get_first),
    issuing_agency_2=('Issuing Agency', get_second),
    issuing_agency_3=('Issuing Agency', get_third),
    penalty_imposed=('Penalty Imposed', 'mean'),
    charge_1 = ('Charge: Code', get_first),
    charge_2 = ('Charge: Code', get_second),
    charge_3 = ('Charge: Code', get_third),
    crime_count = ('Issuing Agency', 'size')
).reset_index()

crime_data_final.rename(columns={'Violation Location (Zip Code)': 'zipcode'}, inplace=True)

crime_data_final.drop(crime_data_final[~crime_data_final['zipcode'].astype(str).apply(lambda x: x.isdigit())].index, inplace=True)

print(crime_data_final['zipcode'].unique())

crime_data_final['zipcode'] = crime_data_final['zipcode'].astype(int)

crime_data_final.head()

['0' '0000' '00000' ... '99337' '99344' '99999']


,zipcode,issuing_agency_1,issuing_agency_2,issuing_agency_3,penalty_imposed,charge_1,charge_2,charge_3,crime_count
0,0,PCS - DOHMH,DOS - ENFORCEMENT AGENTS,SANITATION DEPT,355.681818,AH3P,AS30,AS06,22
1,0,TAXI_TLC,TAXI_TLC,TAXI_TLC,0.000000,19-5,19-5,19-5,4
2,0,TAXI_TLC,TAXI_TLC,TAXI_TLC,0.000000,19-5,19-5,19-5,14
3,0,FIRE DEPARTMENT OF NYC,FIRE DEPARTMENT OF NYC,FIRE DEPARTMENT OF NYC,NaN,BF06,BF06,BF06,1
4,83,PARKS AND RECR,PARKS AND RECR,PARKS AND RECR,35.000000,AA35,AA21,AA75,5


In [26]:
encoder = OrdinalEncoder()

# encode the three species columns
crime_encoded_df = crime_data_final
crime_encoded_df[['issuing_agency_1', 'issuing_agency_2', 'issuing_agency_3']] = encoder.fit_transform(
    crime_data_final[['issuing_agency_1', 'issuing_agency_2', 'issuing_agency_3']])

crime_encoded_df[['charge_1', 'charge_2', 'charge_3']] = encoder.fit_transform(
    crime_data_final[['charge_1', 'charge_2', 'charge_3']])


In [27]:
crime_encoded_df

,zipcode,issuing_agency_1,issuing_agency_2,issuing_agency_3,penalty_imposed,charge_1,charge_2,charge_3,crime_count
0,0,14.0,12.0,21.0,355.681818,95.0,116.0,107.0,22
1,0,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,4
2,0,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,14
3,0,11.0,13.0,16.0,NaN,113.0,125.0,121.0,1
4,83,12.0,14.0,18.0,35.000000,88.0,99.0,97.0,5
...,...,...,...,...,...,...,...,...,...
2937,993210000,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,2
2938,99336,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,1
2939,99337,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,1
2940,99344,19.0,22.0,25.0,0.000000,53.0,57.0,57.0,1


In [28]:
# combining datasets for ML algs
tree_final = tree_encoded_df
crime_final = crime_encoded_df

merged_df = pd.merge(tree_final, crime_final, on='zipcode', how='inner')

# scaled data
scaler = StandardScaler()
scaled_df = pd.DataFrame(scaler.fit_transform(merged_df))
scaled_df.columns = merged_df.columns

scaled_df

,zipcode,dbh_mean,curb_ratio,sidewalk_ratio,species_1,species_2,species_3,tree_count,Good,Fair,Poor,issuing_agency_1,issuing_agency_2,issuing_agency_3,penalty_imposed,charge_1,charge_2,charge_3,crime_count
0,-11.044430,2.004250,-0.793992,-0.087613,-2.070803,1.332725,0.423853,-0.903104,-0.835969,-1.100991,-1.064694,0.473633,-0.137858,0.159386,-2.795917,0.269405,0.374596,0.547784,-1.426030
1,-0.788446,-1.221814,0.280780,0.006109,0.797804,-1.348282,-1.091882,-0.932301,-0.907957,-0.959347,-0.882685,-0.826265,1.272603,-0.146375,0.609955,-0.206723,-2.060006,-1.210338,1.554026
2,-0.787412,-0.835886,-0.698781,0.009415,-0.841400,0.601541,0.423853,-0.466575,-0.501615,-0.292922,-0.217956,-0.826265,0.567373,0.770908,0.681624,-1.516078,0.682125,-1.956208,1.587717
3,-0.786378,-0.617751,1.363982,-0.123738,0.797804,-1.348282,0.423853,-0.543128,-0.577940,-0.353295,-0.320831,-0.826265,0.567373,-0.146375,0.540107,-1.516078,-2.060006,-0.917318,2.535932
4,-0.785344,-1.471641,-0.748432,1.574016,0.797804,0.114085,-1.091882,-1.193293,-1.177696,-1.152076,-1.104261,-0.826265,-0.314165,-2.133821,0.519383,0.239647,-1.368066,-1.956208,-0.759008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,0.959146,-0.710897,-0.173555,-0.006936,0.797804,-0.860826,0.207319,0.674241,0.411247,1.729571,1.586307,0.148659,0.567373,0.770908,-0.697822,0.775292,0.682125,0.867442,-0.114663
184,0.960180,-2.069999,-0.536835,0.090466,-1.661002,-0.129642,-0.009214,-0.571613,-0.733191,0.022875,0.597128,1.286070,-0.490473,-1.369419,-0.675562,0.775292,0.759007,0.867442,-1.121189
185,0.961214,-1.034221,-0.659113,0.553567,-0.841400,0.601541,-1.308415,-0.874264,-0.995124,-0.522804,0.510080,-0.826265,0.743680,-0.299255,-0.482817,0.775292,0.682125,0.867442,-1.107583
186,0.962248,-0.978977,-0.409380,0.112678,0.797804,-1.348282,0.207319,-0.104106,-0.370648,0.742706,1.918672,-0.826265,0.567373,-1.675180,0.509361,0.656260,0.759007,-1.956208,-1.103047


In [29]:
# ML algs
